# 💧 Water Quality Analysis — Notebook Completo
### Análisis de calidad del agua con variable respuesta: **pH**

Este notebook replica el dashboard modular en formato Jupyter.  
Contiene: configuración, EDA, entrenamiento del modelo y predicción interactiva.

---

## 📦 Celda 0 — Instalación de dependencias
Ejecuta esta celda primero (solo la primera vez o en Colab).

In [ ]:
# Instalar librerías necesarias
# En Colab: quita el # de la línea siguiente
# !pip install dash dash-bootstrap-components plotly pandas scikit-learn joblib numpy ipywidgets

# Verificar que todo está disponible
import importlib
libs = ['pandas', 'numpy', 'plotly', 'sklearn', 'joblib']
for lib in libs:
    try:
        importlib.import_module(lib)
        print(f'✅ {lib}')
    except ImportError:
        print(f'❌ {lib} — instalar con pip')

---
## 1️⃣ Carga y exploración inicial del dataset

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Ajusta la ruta según donde tengas el CSV ──────────────────────────────
# Local:  DATA_PATH = 'data/Water_Quality_Testing.csv'
# Colab:  DATA_PATH = '/content/water_quality_dashboard/data/Water_Quality_Testing.csv'
DATA_PATH = 'data/Water_Quality_Testing.csv'

# Cargar y renombrar columnas
df = pd.read_csv(DATA_PATH)
df.columns = ['sample_id', 'ph', 'temperature_c', 'turbidity_ntu',
               'dissolved_oxygen', 'conductivity_us']

# Crear variable categórica de pH
df['ph_category'] = df['ph'].apply(
    lambda x: 'Neutro/Alcalino' if x >= 7.0 else 'Ácido'
)

print(f'Filas: {df.shape[0]} | Columnas: {df.shape[1]}')
print(f'\nCategorías de pH:')
print(df['ph_category'].value_counts())
df.head(10)

In [ ]:
# Estadísticas descriptivas completas
desc = df.drop(columns=['sample_id']).describe().round(3)
desc

In [ ]:
# Verificar valores nulos y tipos
print('=== Valores nulos ===')
print(df.isnull().sum())
print('\n=== Tipos de datos ===')
print(df.dtypes)

---
## 2️⃣ EDA — Análisis Exploratorio de Datos

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Paleta pastel coherente con el dashboard
PASTEL   = ['#AED9E0', '#FFB7B2', '#B5EAD7', '#FFDAC1', '#C7CEEA']
NUMERICS = ['ph', 'temperature_c', 'turbidity_ntu', 'dissolved_oxygen', 'conductivity_us']
LABELS   = {
    'ph'              : 'pH',
    'temperature_c'   : 'Temperatura (°C)',
    'turbidity_ntu'   : 'Turbidez (NTU)',
    'dissolved_oxygen': 'Oxígeno Disuelto (mg/L)',
    'conductivity_us' : 'Conductividad (µS/cm)',
    'ph_category'     : 'Categoría pH',
}
print('✅ Librerías de visualización listas')

In [ ]:
# ── Gráfico de Dona: distribución de categorías de pH ────────────────────
counts = df['ph_category'].value_counts().reset_index()
counts.columns = ['Categoría', 'Cantidad']

fig_donut = go.Figure(go.Pie(
    labels=counts['Categoría'],
    values=counts['Cantidad'],
    hole=0.55,
    marker_colors=[PASTEL[0], PASTEL[1]],
    textinfo='percent+label',
    hovertemplate='%{label}<br>%{value} muestras<extra></extra>',
))
fig_donut.update_layout(
    title='Distribución de Categorías de pH',
    annotations=[dict(text='pH', x=0.5, y=0.5, font_size=22, showarrow=False)],
    width=500, height=420,
)
fig_donut.show()

In [ ]:
# ── Barplot: promedio de variables por categoría de pH ───────────────────
means = df.groupby('ph_category')[NUMERICS].mean().reset_index()

fig_bar = go.Figure()
for i, var in enumerate(NUMERICS):
    fig_bar.add_trace(go.Bar(
        name=LABELS[var],
        x=means['ph_category'],
        y=means[var],
        marker_color=PASTEL[i % len(PASTEL)],
    ))
fig_bar.update_layout(
    title='Promedio de Variables por Categoría de pH',
    barmode='group',
    xaxis_title='Categoría de pH',
    yaxis_title='Valor Promedio',
    legend=dict(orientation='h', y=1.12),
    width=750, height=450,
)
fig_bar.show()

In [ ]:
# ── Histogramas de todas las variables (panel 2x3) ───────────────────────
fig_hist = make_subplots(
    rows=2, cols=3,
    subplot_titles=[LABELS[v] for v in NUMERICS],
)
positions = [(1,1),(1,2),(1,3),(2,1),(2,2)]

for idx, (var, (r, c)) in enumerate(zip(NUMERICS, positions)):
    for cat, color in zip(['Neutro/Alcalino', 'Ácido'], [PASTEL[0], PASTEL[1]]):
        subset = df[df['ph_category'] == cat][var]
        fig_hist.add_trace(
            go.Histogram(
                x=subset, name=cat,
                marker_color=color,
                opacity=0.72,
                showlegend=(idx == 0),
                nbinsx=20,
            ),
            row=r, col=c
        )

fig_hist.update_layout(
    title='Distribuciones por Variable y Categoría de pH',
    barmode='overlay',
    height=550, width=850,
    legend=dict(orientation='h', y=1.05),
)
fig_hist.show()

In [ ]:
# ── Heatmap de correlación ───────────────────────────────────────────────
corr   = df[NUMERICS].corr().round(2)
labels = [LABELS[c] for c in corr.columns]

fig_corr = go.Figure(go.Heatmap(
    z=corr.values,
    x=labels, y=labels,
    colorscale='RdBu',
    zmid=0,
    text=corr.values.round(2),
    texttemplate='%{text}',
    hovertemplate='%{y} – %{x}<br>r = %{z}<extra></extra>',
))
fig_corr.update_layout(
    title='Matriz de Correlación entre Variables',
    width=600, height=500,
)
fig_corr.show()

In [ ]:
# ── Boxplots por categoría ───────────────────────────────────────────────
fig_box = make_subplots(rows=1, cols=len(NUMERICS),
                         subplot_titles=[LABELS[v] for v in NUMERICS])

for idx, var in enumerate(NUMERICS, 1):
    for cat, color in zip(['Neutro/Alcalino', 'Ácido'], [PASTEL[0], PASTEL[1]]):
        fig_box.add_trace(
            go.Box(
                y=df[df['ph_category'] == cat][var],
                name=cat, marker_color=color,
                showlegend=(idx == 1),
            ),
            row=1, col=idx
        )

fig_box.update_layout(
    title='Boxplots por Variable y Categoría de pH',
    height=420, width=950,
    legend=dict(orientation='h', y=1.1),
    boxmode='group',
)
fig_box.show()

---
## 3️⃣ Modelo — Entrenamiento de Regresión Logística

In [ ]:
from sklearn.linear_model    import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import StandardScaler
from sklearn.pipeline        import Pipeline
from sklearn.metrics         import (accuracy_score, precision_score,
                                      recall_score, f1_score,
                                      confusion_matrix, roc_auc_score,
                                      classification_report)

# ── Preparar datos ────────────────────────────────────────────────────────
FEATURES = ['ph', 'temperature_c', 'turbidity_ntu',
            'dissolved_oxygen', 'conductivity_us']
TARGET   = 'ph_category'

X = df[FEATURES]
y = (df[TARGET] == 'Neutro/Alcalino').astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# ── Pipeline: scaler + modelo ─────────────────────────────────────────────
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=500, random_state=42)),
])
pipeline.fit(X_train, y_train)

y_pred  = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

print('=== Reporte de Clasificación ===')
print(classification_report(y_test, y_pred,
      target_names=['Ácido (0)', 'Neutro/Alcalino (1)']))
print(f'ROC-AUC : {roc_auc_score(y_test, y_proba):.4f}')

In [ ]:
# ── Tarjetas de métricas ─────────────────────────────────────────────────
metrics = {
    'Accuracy' : accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall'   : recall_score(y_test, y_pred),
    'F1-Score' : f1_score(y_test, y_pred),
    'ROC-AUC'  : roc_auc_score(y_test, y_proba),
}

fig_metrics = go.Figure()
colors = PASTEL
for i, (name, val) in enumerate(metrics.items()):
    fig_metrics.add_trace(go.Bar(
        x=[name], y=[val],
        marker_color=colors[i % len(colors)],
        text=[f'{val:.1%}'], textposition='outside',
        name=name,
    ))
fig_metrics.update_layout(
    title='Métricas del Modelo',
    yaxis=dict(range=[0, 1.15], tickformat='.0%'),
    showlegend=False,
    height=400, width=600,
)
fig_metrics.show()

In [ ]:
# ── Curva ROC ─────────────────────────────────────────────────────────────
from sklearn.metrics import roc_curve, auc

fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc_val = auc(fpr, tpr)

fig_roc = go.Figure()
fig_roc.add_trace(go.Scatter(
    x=fpr, y=tpr, mode='lines',
    name=f'ROC (AUC = {roc_auc_val:.3f})',
    line=dict(color='#AED9E0', width=3),
    fill='tozeroy', fillcolor='rgba(174,217,224,0.2)',
))
fig_roc.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode='lines',
    line=dict(color='gray', width=1.5, dash='dash'),
    name='Clasificador aleatorio',
))
fig_roc.update_layout(
    title='Curva ROC',
    xaxis_title='Tasa de Falsos Positivos (FPR)',
    yaxis_title='Tasa de Verdaderos Positivos (TPR)',
    width=560, height=440,
    legend=dict(x=0.55, y=0.1),
)
fig_roc.show()

In [ ]:
# ── Matriz de confusión ───────────────────────────────────────────────────
cm     = confusion_matrix(y_test, y_pred)
labels = ['Ácido (0)', 'Neutro/Alcalino (1)']

fig_cm = go.Figure(go.Heatmap(
    z=cm,
    x=labels, y=labels,
    colorscale=[[0, '#f8f9fa'], [1, '#AED9E0']],
    showscale=False,
    text=[[str(v) for v in row] for row in cm],
    texttemplate='<b>%{text}</b>',
    hovertemplate='Real: %{y}<br>Predicho: %{x}<br>Conteo: %{z}<extra></extra>',
))
fig_cm.update_layout(
    title='Matriz de Confusión',
    xaxis_title='Predicho',
    yaxis_title='Real',
    width=480, height=420,
)
fig_cm.show()

In [ ]:
# ── Importancia de variables (coeficientes del modelo) ────────────────────
coefs = pipeline.named_steps['clf'].coef_[0]
feat_importance = pd.DataFrame({
    'Variable'   : [LABELS[f] for f in FEATURES],
    'Coeficiente': coefs,
    'Abs'        : np.abs(coefs),
}).sort_values('Abs', ascending=True)

fig_coef = go.Figure(go.Bar(
    x=feat_importance['Coeficiente'],
    y=feat_importance['Variable'],
    orientation='h',
    marker_color=[
        '#AED9E0' if v >= 0 else '#FFB7B2'
        for v in feat_importance['Coeficiente']
    ],
))
fig_coef.update_layout(
    title='Coeficientes del Modelo (importancia de variables)',
    xaxis_title='Coeficiente',
    width=600, height=380,
)
fig_coef.show()

---
## 4️⃣ Predicción Interactiva con ipywidgets
Mueve los sliders y presiona **Predecir** para clasificar una nueva muestra.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ── Sliders de entrada ────────────────────────────────────────────────────
sl_ph   = widgets.FloatSlider(value=7.2, min=6.5,  max=8.5,  step=0.01,
                               description='pH:',
                               style={'description_width': '160px'},
                               layout=widgets.Layout(width='500px'))
sl_temp = widgets.FloatSlider(value=25.0, min=15.0, max=35.0, step=0.1,
                               description='Temperatura (°C):',
                               style={'description_width': '160px'},
                               layout=widgets.Layout(width='500px'))
sl_turb = widgets.FloatSlider(value=3.0, min=0.5,  max=10.0, step=0.1,
                               description='Turbidez (NTU):',
                               style={'description_width': '160px'},
                               layout=widgets.Layout(width='500px'))
sl_do   = widgets.FloatSlider(value=8.0, min=5.0,  max=12.0, step=0.1,
                               description='Oxígeno Disuelto (mg/L):',
                               style={'description_width': '160px'},
                               layout=widgets.Layout(width='500px'))
sl_cond = widgets.IntSlider(value=350, min=200, max=600, step=5,
                             description='Conductividad (µS/cm):',
                             style={'description_width': '160px'},
                             layout=widgets.Layout(width='500px'))

btn     = widgets.Button(description='🔍 Predecir',
                          button_style='primary',
                          layout=widgets.Layout(width='200px', height='40px'))
output  = widgets.Output()

def predecir(b):
    with output:
        clear_output(wait=True)
        X_new   = np.array([[sl_ph.value, sl_temp.value,
                              sl_turb.value, sl_do.value, sl_cond.value]])
        pred    = pipeline.predict(X_new)[0]
        proba   = pipeline.predict_proba(X_new)[0]
        prob_1  = proba[1]

        if pred == 1:
            icono, etiqueta, bg, color = '✅', 'Neutro / Alcalino', '#d4edda', '#155724'
        else:
            icono, etiqueta, bg, color = '⚠️', 'Ácido', '#f8d7da', '#721c24'

        html_out = f"""
        <div style="background:{bg};border-radius:12px;padding:20px 30px;
                    font-family:sans-serif;max-width:480px;margin-top:10px">
          <h2 style="color:{color};margin:0">{icono} {etiqueta}</h2>
          <hr style="border-color:{color};opacity:0.3">
          <p style="margin:4px 0"><b>Probabilidad Neutro/Alcalino:</b> {prob_1:.1%}</p>
          <p style="margin:4px 0"><b>Probabilidad Ácido:</b>            {1-prob_1:.1%}</p>
          <hr style="border-color:{color};opacity:0.3">
          <p style="margin:4px 0;font-size:13px">
            pH={sl_ph.value} | Temp={sl_temp.value}°C |
            Turb={sl_turb.value} NTU | O₂={sl_do.value} mg/L |
            Cond={sl_cond.value} µS/cm
          </p>
        </div>
        """
        display(HTML(html_out))

        # Mini gauge con plotly
        gauge = go.Figure(go.Indicator(
            mode='gauge+number',
            value=prob_1 * 100,
            title={'text': 'Probabilidad Neutro/Alcalino (%)'},
            gauge={
                'axis' : {'range': [0, 100]},
                'bar'  : {'color': '#AED9E0'},
                'steps': [
                    {'range': [0,  50], 'color': '#FFB7B2'},
                    {'range': [50, 100], 'color': '#B5EAD7'},
                ],
                'threshold': {'line': {'color': '#333', 'width': 4},
                              'thickness': 0.75, 'value': 50},
            },
            number={'suffix': '%'},
        ))
        gauge.update_layout(height=280, width=400,
                            margin=dict(t=60, b=10, l=20, r=20))
        gauge.show()

btn.on_click(predecir)

display(
    widgets.VBox([
        widgets.HTML('<h3 style="font-family:sans-serif">⚙️ Parámetros de la muestra</h3>'),
        sl_ph, sl_temp, sl_turb, sl_do, sl_cond,
        btn, output,
    ])
)

---
## 5️⃣ Guardar y cargar el modelo (.pkl)
Si quieres guardar el modelo entrenado para usarlo en el dashboard Dash:

In [ ]:
import joblib, os

# Guardar
MODEL_PATH = 'model/model.pkl'
os.makedirs('model', exist_ok=True)

joblib.dump({
    'pipeline': pipeline,
    'metrics' : metrics,
}, MODEL_PATH)
print(f'✅ Modelo guardado en {MODEL_PATH}')

# Verificar que se puede cargar
data_loaded = joblib.load(MODEL_PATH)
print(f'✅ Modelo cargado correctamente')
print(f'   Accuracy en test: {data_loaded["metrics"]["Accuracy"]:.1%}')

---
## 6️⃣ Lanzar el Dashboard Dash desde el notebook
Si prefieres ver el dashboard completo sin salir de Jupyter/Colab:

In [ ]:
# ── OPCIÓN A: Local (VS Code / Jupyter Lab) ───────────────────────────────
# Corre esto y abre localhost:8050 en el navegador

import subprocess, threading, time

def run_dash():
    subprocess.run(['python', 'app.py'])

threading.Thread(target=run_dash, daemon=True).start()
time.sleep(2)
print('✅ Dashboard corriendo en http://localhost:8050')

In [ ]:
# ── OPCIÓN B: Google Colab con ngrok ─────────────────────────────────────
# Descomenta y pega tu token de ngrok.com

# import subprocess, threading, time
# from pyngrok import ngrok, conf
#
# ngrok.kill()   # cerrar túneles anteriores
# time.sleep(2)
#
# conf.get_default().auth_token = 'PEGA_TU_TOKEN_AQUI'
# public_url = ngrok.connect(8050)
# print('🌐 Dashboard en:', public_url)
#
# def run_dash():
#     subprocess.run(['python', 'app.py'], cwd='/content/water_quality_dashboard')
#
# threading.Thread(target=run_dash, daemon=True).start()

---
### 📌 Resumen del proyecto

| Elemento | Detalle |
|---|---|
| Dataset | Water Quality Testing — 500 muestras, 5 variables |
| Variable respuesta | pH → Ácido (< 7.0) / Neutro-Alcalino (≥ 7.0) |
| Modelo | Regresión Logística con StandardScaler |
| Split | 75% entrenamiento / 25% prueba |
| Dashboard | Dash + Bootstrap — 3 pestañas: Contexto, EDA, Metodología |

> **Nota:** Para ver el dashboard completo con todas las pestañas, ejecuta `python app.py` en la terminal.